# Notebook 03: Parameter Extraction

**Objective:** Run global optimization + local refinement to extract SPICE parameters from I-V data.

We cover:
1. Generate synthetic target data with known parameters
2. Build multi-curve objective function
3. Run DE (global) + TRF (local) two-stage extraction
4. Compare extracted vs true parameters
5. Visualize fitting quality

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt

from src.device.mosfet import MOSFETLevel3, MOSFETParamsLevel3
from src.device.curves import generate_iv_curves
from src.extraction.objective import (
    ExtractionObjective, CurveData, params_to_vector, vector_to_params
)
from src.extraction.optimizer import two_stage_extraction
from src.viz.plots import plot_extraction_comparison, plot_optimization_trace

## 1. Generate Target Data

We create a MOSFET with known parameters to use as "ground truth".

In [ ]:
# Randomize target parameters
rng = np.random.RandomState(42)
p_true = MOSFETParamsLevel3(
    VTH0=rng.uniform(0.3, 0.7),
    U0=rng.uniform(200, 500),
    THETA=rng.uniform(0.02, 0.10),
    VSAT=rng.uniform(6e6, 12e6),
    ETA0=rng.uniform(0.03, 0.08),
)

model_true = MOSFETLevel3(p_true)
id_vg, id_vd = generate_iv_curves(model_true)

print("True parameters:")
for name in ['VTH0', 'U0', 'THETA', 'VSAT', 'ETA0']:
    print(f"  {name:10s} = {getattr(p_true, name):.6g}")

## 2. Build Extraction Objective

In [ ]:
tg_vg = CurveData(vgs=id_vg.vgs, vds=id_vg.vds_values, ids=id_vg.ids)
tg_vd = CurveData(vgs=id_vd.vgs_values, vds=id_vd.vds, ids=id_vd.ids)

objective = ExtractionObjective(
    target_idvg=tg_vg,
    target_idvd=tg_vd,
    weights=(1.0, 0.3, 0.2),
)

print(f"Objective at true params: {objective(params_to_vector(p_true), MOSFETLevel3):.6e}")

## 3. Run Two-Stage Extraction

- **Stage 1:** Differential Evolution (50 iterations, global coarse search)
- **Stage 2:** TRF (local fine-tuning)

In [ ]:
result = two_stage_extraction(
    objective,
    stage1="de",
    stage1_options={"max_iter": 80, "pop_size": 20},
    verbose=True,
)

## 4. Compare Extracted vs True Parameters

In [ ]:
x_ext = result["x_refined"]
param_names = result["param_names"]

print(f"{'Parameter':<12s} {'True':>12s} {'Extracted':>12s} {'Error%':>8s}")
print("-" * 50)
errors = []
for i, name in enumerate(param_names):
    if hasattr(p_true, name):
        true_val = getattr(p_true, name)
        ext_val = x_ext[i]
        err = abs(true_val - ext_val) / max(abs(true_val), 1e-12) * 100
        errors.append(err)
        print(f"{name:<12s} {true_val:>12.4g} {ext_val:>12.4g} {err:>7.2f}%")

n_good = sum(1 for e in errors if e < 5.0)
print(f"\nParameters within 5% error: {n_good}/{len(errors)}")
print(f"Mean error: {np.mean(errors):.2f}%")

## 5. Visualize Fit Quality

In [ ]:
# Generate curves from extracted parameters
p_ext = vector_to_params(x_ext, MOSFETLevel3)
model_ext = MOSFETLevel3(p_ext)
ext_vg, ext_vd = generate_iv_curves(model_ext)

fig = plot_extraction_comparison(id_vg, ext_vg, vds_idx=-1,
                                 title="Target vs Extracted")
plt.show()

## 6. Optimization Convergence

In [ ]:
if "history" in result.get("stage1_info", {}):
    fig = plot_optimization_trace(result["stage1_info"]["history"])
    plt.show()
else:
    print("DE optimizer doesn't provide per-iteration history.")